# SageMaker Model Bias Monitor

This notebook adds a SageMaker Clarify Model Bias Monitor to the NYC Motor Vehicle Collisions project.

The existing monitoring notebook already includes Data Quality Monitoring, Model Quality Monitoring, and a CloudWatch dashboard. This notebook adds the missing bias monitoring component.

The model predicts whether a crash results in injury. The target column is `target`, where `1` means an injury occurred and `0` means no injury occurred.

The selected bias facet is `borough`, because prediction behavior may vary across NYC boroughs.

In [ ]:
!pip install "sagemaker==2.214.0"

import boto3
import sagemaker
import pandas as pd
import botocore

from sagemaker import get_execution_role
from sagemaker.model_monitor import (
    ModelBiasMonitor,
    CronExpressionGenerator,
    EndpointInput
)
from sagemaker.clarify import (
    BiasConfig,
    DataConfig,
    ModelConfig,
    ModelPredictedLabelConfig
)

sess = sagemaker.Session()
role = get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name
sm_client = boto3.client("sagemaker", region_name=region)

print("Region:", region)
print("Bucket:", bucket)

## Find the Live Endpoint

Notebook 05 creates a live SageMaker endpoint with a name that starts with `collisions-monitor-ep`. This section finds the newest matching endpoint so the bias monitor can attach to it.

In [ ]:
endpoint_name = None

endpoints = sm_client.list_endpoints(
    SortBy="CreationTime",
    SortOrder="Descending"
)

for ep in endpoints["Endpoints"]:
    if ep["EndpointName"].startswith("collisions-monitor-ep"):
        endpoint_name = ep["EndpointName"]
        break

if endpoint_name is None:
    raise ValueError("No endpoint starting with 'collisions-monitor-ep' found. Run notebook 05 first.")

print("Using endpoint:", endpoint_name)

## Bias Monitoring Configuration

The bias monitor uses the validation dataset from notebook 04 as the baseline.

Where we use the following:

- Label column: `target`
- Positive label: `1`
- Bias facet: `borough`
- Endpoint: live endpoint created in notebook 05

In [ ]:
monitoring_prefix = "nyc-collisions-monitoring"
training_prefix = "aai-540-group6/nyc-collisions-ml"

baseline_data_uri = f"s3://{bucket}/{training_prefix}/validation/val.csv"
bias_output_uri = f"s3://{bucket}/{monitoring_prefix}/model_bias_results"

label_column = "target"
facet_name = "borough"

df_val = pd.read_csv("data_splits/val.csv")
headers = df_val.columns.tolist()

print("Validation shape:", df_val.shape)
print("Baseline S3 path:", baseline_data_uri)
print("Bias output path:", bias_output_uri)
print("Columns:", headers)

if label_column not in df_val.columns:
    raise ValueError(f"Missing label column: {label_column}")

if facet_name not in df_val.columns:
    raise ValueError(f"Missing facet column: {facet_name}")

## Create Model Bias Baseline

SageMaker Clarify first creates a baseline from validation data. This baseline is later used to compare future endpoint predictions and detect bias drift.

In [ ]:
bias_config = BiasConfig(
    label_values_or_threshold=[1],
    facet_name=facet_name
)

data_config = DataConfig(
    s3_data_input_path=baseline_data_uri,
    s3_output_path=f"{bias_output_uri}/baseline",
    label=label_column,
    headers=headers,
    dataset_type="text/csv"
)

model_config = ModelConfig(
    model_name=endpoint_name,
    instance_type="ml.m5.large",
    instance_count=1,
    accept_type="text/csv",
    content_type="text/csv"
)

predicted_label_config = ModelPredictedLabelConfig(
    label="0"
)

model_bias_monitor = ModelBiasMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    max_runtime_in_seconds=1800,
    sagemaker_session=sess
)

model_bias_monitor.suggest_baseline(
    data_config=data_config,
    bias_config=bias_config,
    model_config=model_config,
    model_predicted_label_config=predicted_label_config,
    wait=True
)

print("Model Bias baseline completed.")

## Create Bias Monitoring Schedule

This section creates an hourly monitoring schedule. The monitor checks captured endpoint predictions and compares them against the bias baseline.

In [ ]:
bias_schedule_name = f"{endpoint_name}-bias-monitor"

try:
    sm_client.describe_monitoring_schedule(
        MonitoringScheduleName=bias_schedule_name
    )
    print(f"Schedule already exists: {bias_schedule_name}")

except botocore.exceptions.ClientError:
    model_bias_monitor.create_monitoring_schedule(
        monitor_schedule_name=bias_schedule_name,
        endpoint_input=EndpointInput(
            endpoint_name=endpoint_name,
            destination="/opt/ml/processing/input_data",
            inference_attribute="0"
        ),
        output_s3_uri=f"{bias_output_uri}/schedule",
        statistics=model_bias_monitor.baseline_statistics(),
        constraints=model_bias_monitor.suggested_constraints(),
        schedule_cron_expression=CronExpressionGenerator.hourly(),
        enable_cloudwatch_metrics=True
    )

    print("Model Bias monitoring schedule created:", bias_schedule_name)